In [ ]:
%pip install -q kagglehub lightgbm catboost xgboost scikit-learn torch pandas numpy matplotlib

# 🏏 IPL 2026 Match Winner Prediction

**Models:** Logistic Regression · Random Forest · XGBoost · LightGBM · CatBoost · Neural Network (PyTorch) · Stacking Ensemble
**Dataset:** [IPL Ball-by-Ball Data 2008–2025](https://www.kaggle.com/datasets/chaitu20/ipl-dataset2008-2025) (Kaggle)
**Target:** Predict whether the team batting first (`team1`) wins the match

The script follows a leakage-safe workflow: features are computed using only matches
played *before* each match, and train/test are split by time (train ≤ 2022, test ≥ 2023).

Run as a script (`python ipl_2026_match_winner_prediction.py`) or open the matching
`.ipynb` notebook. Plots and tables are saved to `outputs/`.

## 0. Configuration & Imports

In [ ]:
import os
import random
import time
import warnings
from collections import defaultdict

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ---- Config -----------------------------------------------------------------
SEED = 42                 # set to None for non-reproducible runs
N_SIMULATIONS = 10_000    # Monte Carlo season simulations
NN_EPOCHS = 300
TRAIN_END_SEASON = 2022   # train: <= 2022, test: >= 2023
OUTPUT_DIR = "outputs"
KAGGLE_DATASET = "chaitu20/ipl-dataset2008-2025"
# Optional: point to a local folder containing IPL.csv to skip the Kaggle download
LOCAL_DATA_DIR = os.environ.get("IPL_DATA_DIR")
# -----------------------------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)
if SEED is not None:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

## 1. Data Loading

In [ ]:
path = LOCAL_DATA_DIR or kagglehub.dataset_download(KAGGLE_DATASET)
print("Files:", os.listdir(path))

df = pd.read_csv(os.path.join(path, "IPL.csv"), low_memory=False)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

## 2. Preprocessing — Build Match-Level Dataset

In [ ]:
# Aggregate ball-by-ball -> one row per match
match_cols = [
    "match_id", "date", "venue", "city",
    "batting_team", "bowling_team",
    "toss_winner", "toss_decision",
    "match_won_by", "win_outcome", "stage",
]
match_df = df.groupby("match_id").first().reset_index()[match_cols].copy()
print("Match-level shape:", match_df.shape)

In [ ]:
# Standardise team names (franchise renames) and drop defunct teams (None)
abbrev_map = {
    "Chennai Super Kings": "CSK",
    "Delhi Capitals": "DC",
    "Delhi Daredevils": "DC",
    "Gujarat Titans": "GT",
    "Kolkata Knight Riders": "KKR",
    "Mumbai Indians": "MI",
    "Punjab Kings": "PBKS",
    "Kings XI Punjab": "PBKS",
    "Rajasthan Royals": "RR",
    "Royal Challengers Bangalore": "RCB",
    "Royal Challengers Bengaluru": "RCB",
    "Sunrisers Hyderabad": "SRH",
    "Lucknow Super Giants": "LSG",
    # Defunct franchises -> excluded
    "Rising Pune Supergiant": None,
    "Rising Pune Supergiants": None,
    "Deccan Chargers": None,
    "Kochi Tuskers Kerala": None,
    "Pune Warriors": None,
    "Gujarat Lions": None,
}

for col in ["batting_team", "bowling_team", "toss_winner", "match_won_by"]:
    match_df[col] = match_df[col].map(abbrev_map)

# Keep only matches between current (2026) IPL teams with a decisive result
ipl_2026_teams = {"CSK", "DC", "GT", "KKR", "LSG", "MI", "PBKS", "RCB", "RR", "SRH"}
mask = (
    match_df["batting_team"].isin(ipl_2026_teams)
    & match_df["bowling_team"].isin(ipl_2026_teams)
    & match_df["match_won_by"].isin(ipl_2026_teams)
)
match_df = match_df[mask].reset_index(drop=True)
print("Matches after filtering:", len(match_df))
print("Teams:", sorted(match_df["batting_team"].unique()))

In [ ]:
# Season = calendar year of the match date. This is robust to the mixed
# '2007/08', '2009/10', '2020/21' labels in the raw `season` column, and keeps
# the 2020 (UAE) and 2021 seasons separate.
match_df["date"] = pd.to_datetime(match_df["date"])
match_df["season"] = match_df["date"].dt.year
print("Seasons:", sorted(match_df["season"].unique()))

## 3. Feature Engineering

Every rolling feature for a match uses **only earlier matches** (`match_df.iloc[:i]`).

In [ ]:
# Sort chronologically — critical to prevent data leakage
match_df = match_df.sort_values(["date", "match_id"]).reset_index(drop=True)

match_df["team1"] = match_df["batting_team"]    # bats first
match_df["team2"] = match_df["bowling_team"]    # chases
match_df["team1_won"] = (match_df["match_won_by"] == match_df["team1"]).astype(int)

knockout_stages = {"Final", "Qualifier 1", "Qualifier 2", "Eliminator", "Semi Final", "Elimination Final"}
match_df["is_knockout"] = match_df["stage"].isin(knockout_stages).astype(int)
print("Base features done.")

In [ ]:
def win_rate(team, matches):
    """Win rate of `team` across the given matches (0.5 if no matches)."""
    if len(matches) == 0:
        return 0.5
    wins = (
        ((matches["team1"] == team) & (matches["team1_won"] == 1)).sum()
        + ((matches["team2"] == team) & (matches["team1_won"] == 0)).sum()
    )
    return wins / len(matches)


def streak(team, past):
    """Current streak: +n = n straight wins, -n = n straight losses."""
    games = past[(past["team1"] == team) | (past["team2"] == team)]
    s = 0
    for _, r in games.iloc[::-1].iterrows():
        won = (r["team1"] == team and r["team1_won"] == 1) or (
            r["team2"] == team and r["team1_won"] == 0
        )
        if s == 0:
            s = 1 if won else -1
        elif (s > 0 and won) or (s < 0 and not won):
            s += 1 if won else -1
        else:
            break
    return s


# First-innings total per match (for venue average score)
fi = df[df["innings"] == 1].groupby("match_id")["runs_total"].sum().reset_index()
fi.columns = ["match_id", "fi_runs"]
fi["venue"] = fi["match_id"].map(df.groupby("match_id")["venue"].first())

In [ ]:
print("Computing rolling features (may take a few minutes)...")

feature_rows = []
for i, row in match_df.iterrows():
    past = match_df.iloc[:i]
    t1, t2 = row["team1"], row["team2"]

    m1 = past[(past["team1"] == t1) | (past["team2"] == t1)]
    m2 = past[(past["team1"] == t2) | (past["team2"] == t2)]

    bat1, bat2 = past[past["team1"] == t1], past[past["team1"] == t2]    # batted first
    ch1, ch2 = past[past["team2"] == t1], past[past["team2"] == t2]      # chased

    h = past[
        ((past["team1"] == t1) & (past["team2"] == t2))
        | ((past["team1"] == t2) & (past["team2"] == t1))
    ]
    past_fi = fi[(fi["venue"] == row["venue"]) & (fi["match_id"] < row["match_id"])]

    feature_rows.append({
        "t1_wr": win_rate(t1, m1),
        "t2_wr": win_rate(t2, m2),
        "t1_swr": win_rate(t1, m1[m1["season"] == row["season"]]),
        "t2_swr": win_rate(t2, m2[m2["season"] == row["season"]]),
        "t1_f5": win_rate(t1, m1.tail(5)),
        "t2_f5": win_rate(t2, m2.tail(5)),
        "t1_f10": win_rate(t1, m1.tail(10)),
        "t2_f10": win_rate(t2, m2.tail(10)),
        "t1_bat": bat1["team1_won"].mean() if len(bat1) else 0.5,
        "t2_bat": bat2["team1_won"].mean() if len(bat2) else 0.5,
        "t1_ch": (ch1["team1_won"] == 0).mean() if len(ch1) else 0.5,
        "t2_ch": (ch2["team1_won"] == 0).mean() if len(ch2) else 0.5,
        "t1_vwr": win_rate(t1, m1[m1["venue"] == row["venue"]]),
        "h2h": win_rate(t1, h),
        "t1_str": streak(t1, past),
        "t2_str": streak(t2, past),
        "venue_avg": past_fi["fi_runs"].mean() if len(past_fi) else 150,
    })

match_df = pd.concat([match_df, pd.DataFrame(feature_rows, index=match_df.index)], axis=1)
print("All features computed!")

In [ ]:
# Differential features — cleaner signal for models
match_df["wr_diff"] = match_df["t1_wr"] - match_df["t2_wr"]
match_df["swr_diff"] = match_df["t1_swr"] - match_df["t2_swr"]
match_df["f5_diff"] = match_df["t1_f5"] - match_df["t2_f5"]
match_df["f10_diff"] = match_df["t1_f10"] - match_df["t2_f10"]
match_df["bat_diff"] = match_df["t1_bat"] - match_df["t2_bat"]
match_df["ch_diff"] = match_df["t1_ch"] - match_df["t2_ch"]
match_df["str_diff"] = match_df["t1_str"] - match_df["t2_str"]
match_df["h2h_diff"] = match_df["h2h"] - 0.5

FEATURES = [
    "wr_diff", "swr_diff", "f5_diff", "f10_diff",
    "bat_diff", "ch_diff", "str_diff", "h2h_diff",
    "t1_vwr", "venue_avg", "season", "is_knockout",
]

X = match_df[FEATURES]
y = match_df["team1_won"]

print(f"Features : {len(FEATURES)}")
print(f"Samples  : {len(X)}")
print(f"Nulls    : {X.isnull().sum().sum()}")
print(f"Class balance:\n{y.value_counts(normalize=True).round(3)}")

## 4. Train/Test Split (Time-Based)

Train on seasons ≤ 2022, test on 2023–2025. Random splits would leak future information.

In [ ]:
train = match_df[match_df["season"] <= TRAIN_END_SEASON]
test = match_df[match_df["season"] > TRAIN_END_SEASON]

X_train, y_train = train[FEATURES], train["team1_won"]
X_test, y_test = test[FEATURES], test["team1_won"]

baseline_acc = y_test.value_counts(normalize=True).max()
print(f"Train: {len(X_train)} matches ({train.season.min()}–{train.season.max()})")
print(f"Test : {len(X_test)} matches ({test.season.min()}–{test.season.max()})")
print(f"Baseline accuracy (always predict majority class): {baseline_acc:.3f}")

## 5. Model Training & Evaluation

Cross-validation uses `TimeSeriesSplit` so each fold validates on matches that come
*after* its training data.

In [ ]:
cv = TimeSeriesSplit(n_splits=5)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=0.05, class_weight="balanced", max_iter=1000, random_state=SEED)),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=4, min_samples_split=20,
        min_samples_leaf=10, class_weight="balanced", random_state=SEED,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        random_state=SEED, eval_metric="logloss", verbosity=0,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        class_weight="balanced", random_state=SEED, verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=200, depth=3, learning_rate=0.05,
        auto_class_weights="Balanced", random_seed=SEED, verbose=0,
    ),
}

# Stacking ensemble — uses the models above as base learners
estimators = [
    ("lr", Pipeline([("s", StandardScaler()),
                     ("c", LogisticRegression(C=0.05, class_weight="balanced", max_iter=1000, random_state=SEED))])),
    ("rf", RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                  class_weight="balanced", random_state=SEED)),
    ("xgb", XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05,
                          verbosity=0, random_state=SEED, eval_metric="logloss")),
    ("lgb", LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.05,
                           verbose=-1, random_state=SEED)),
    ("cat", CatBoostClassifier(iterations=100, depth=3, learning_rate=0.05,
                               verbose=0, random_seed=SEED)),
]
models["Stacking"] = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(C=0.1, max_iter=1000, random_state=SEED),
    cv=5, passthrough=True,
)
print("All models defined.")

In [ ]:
results = []

for name, model in models.items():
    print(f"Training {name}...", end=" ")

    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    cv_score = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy").mean()

    try:
        roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    except Exception:
        roc_auc = None

    results.append({
        "Model": name,
        "Train Acc": round(train_acc, 3),
        "CV Acc": round(cv_score, 3),
        "Test Acc": round(test_acc, 3),
        "ROC-AUC": round(roc_auc, 3) if roc_auc is not None else "-",
        "Overfit Gap": round(train_acc - test_acc, 3),
    })
    print(f"Done — Test Acc: {test_acc:.3f}")

results_df = pd.DataFrame(results).sort_values("Test Acc", ascending=False).reset_index(drop=True)
print("\n=== MODEL COMPARISON ===")
print(results_df.to_string(index=False))

## 6. Neural Network (PyTorch)

The reported accuracy is from the **final epoch** (not the best epoch on the test set),
to avoid picking the checkpoint using test data.

In [ ]:
nn_scaler = StandardScaler()
X_train_sc = nn_scaler.fit_transform(X_train)
X_test_sc = nn_scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train_sc)
y_train_t = torch.FloatTensor(y_train.values)
X_test_t = torch.FloatTensor(X_test_sc)
y_test_t = torch.FloatTensor(y_test.values)

loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)


class IPLNet(nn.Module):
    def __init__(self, inp):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(inp, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(16, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


nn_model = IPLNet(X_train_t.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(nn_model.parameters(), lr=0.001, weight_decay=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

train_losses, test_accs = [], []

for epoch in range(NN_EPOCHS):
    nn_model.train()
    epoch_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(nn_model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()

    nn_model.eval()
    with torch.no_grad():
        acc = ((nn_model(X_test_t) > 0.5).float() == y_test_t).float().mean().item()

    train_losses.append(epoch_loss / len(loader))
    test_accs.append(acc)

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:>3} | Loss: {train_losses[-1]:.4f} | Test Acc: {acc:.3f}")

print(f"\nFinal-epoch Test Acc: {test_accs[-1]:.3f}")

In [ ]:
# Add the neural network to the comparison table
nn_model.eval()
with torch.no_grad():
    nn_train_acc = ((nn_model(X_train_t) > 0.5).float() == y_train_t).float().mean().item()
    nn_test_prob = nn_model(X_test_t).numpy()
nn_test_acc = test_accs[-1]

results_df = pd.concat([results_df, pd.DataFrame([{
    "Model": "Neural Network",
    "Train Acc": round(nn_train_acc, 3),
    "CV Acc": "-",
    "Test Acc": round(nn_test_acc, 3),
    "ROC-AUC": round(roc_auc_score(y_test, nn_test_prob), 3),
    "Overfit Gap": round(nn_train_acc - nn_test_acc, 3),
}])], ignore_index=True)

results_df = results_df.sort_values("Test Acc", ascending=False).reset_index(drop=True)
print("=== FINAL MODEL COMPARISON ===")
print(results_df.to_string(index=False))
results_df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"), index=False)

## 7. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("IPL 2026 — Model Comparison", fontsize=14, fontweight="bold")

# Plot 1: test accuracy
colors = ["green" if v == results_df["Test Acc"].max() else "steelblue" for v in results_df["Test Acc"]]
axes[0].barh(results_df["Model"], results_df["Test Acc"], color=colors)
axes[0].axvline(x=baseline_acc, color="red", linestyle="--", label=f"Baseline ({baseline_acc:.3f})")
axes[0].set_xlabel("Test Accuracy")
axes[0].set_title("Test Accuracy by Model")
axes[0].legend()
for i, v in enumerate(results_df["Test Acc"]):
    axes[0].text(v + 0.002, i, f"{v:.3f}", va="center")

# Plot 2: train vs test (overfitting check)
num_models = results_df[results_df["Train Acc"] != "-"]
x = np.arange(len(num_models))
axes[1].bar(x - 0.2, num_models["Train Acc"], 0.4, label="Train", color="steelblue")
axes[1].bar(x + 0.2, num_models["Test Acc"], 0.4, label="Test", color="coral")
axes[1].set_xticks(x)
axes[1].set_xticklabels(num_models["Model"], rotation=30, ha="right")
axes[1].set_title("Train vs Test Accuracy")
axes[1].legend()

# Plot 3: neural network test-accuracy curve
lr_acc = results_df.loc[results_df["Model"] == "Logistic Regression", "Test Acc"].values[0]
axes[2].plot(test_accs, color="steelblue", label="NN Test Accuracy")
axes[2].axhline(y=lr_acc, color="red", linestyle="--", label="Logistic Regression")
axes[2].axhline(y=baseline_acc, color="gray", linestyle=":", label="Baseline")
axes[2].set_xlabel("Epoch")
axes[2].set_title("Neural Network — Test Accuracy Curve")
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Best Model — Detailed Evaluation

> Note: the "best" model is chosen by test accuracy on ~200 matches, so differences of a
> few points are within noise. Treat the ranking as indicative, not definitive.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
print(f"Best Model: {best_model_name}")
print(f"Test Accuracy: {results_df.iloc[0]['Test Acc']}")

if best_model_name == "Neural Network":
    y_pred_best = (nn_test_prob > 0.5).astype(int)
else:
    y_pred_best = models[best_model_name].predict(X_test)

print(f"\n{classification_report(y_test, y_pred_best, target_names=['Team2 wins', 'Team1 wins'])}")

## 9. Predict IPL 2026 Fixtures

`team1` = team batting first. `get_proba` returns `[P(team2 wins), P(team1 wins)]`.

In [ ]:
city_to_venue = {
    "Bengaluru": "M Chinnaswamy Stadium, Bengaluru",
    "Mumbai": "Wankhede Stadium, Mumbai",
    "Guwahati": "ACA Stadium, Guwahati",
    "Mullanpur": "PCA International Cricket Stadium, New Chandigarh",
    "Lucknow": "Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow",
    "Kolkata": "Eden Gardens, Kolkata",
    "Chennai": "MA Chidambaram Stadium, Chennai",
    "Delhi": "Arun Jaitley Stadium, Delhi",
    "Ahmedabad": "Narendra Modi Stadium, Ahmedabad",
    "Hyderabad": "Rajiv Gandhi International Stadium, Hyderabad",
    "Jaipur": "Sawai Mansingh Stadium, Jaipur",
}

team_venues = {
    "CSK": "MA Chidambaram Stadium, Chennai",
    "DC": "Arun Jaitley Stadium, Delhi",
    "GT": "Narendra Modi Stadium, Ahmedabad",
    "KKR": "Eden Gardens, Kolkata",
    "LSG": "Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow",
    "MI": "Wankhede Stadium, Mumbai",
    "PBKS": "PCA International Cricket Stadium, New Chandigarh",
    "RCB": "M Chinnaswamy Stadium, Bengaluru",
    "RR": "Sawai Mansingh Stadium, Jaipur",
    "SRH": "Rajiv Gandhi International Stadium, Hyderabad",
}


# ---- Helpers: current-form features computed from ALL historical matches -----
def get_wr(team):
    m = match_df[(match_df["team1"] == team) | (match_df["team2"] == team)]
    return win_rate(team, m)


def get_swr(team, season=2025):
    m = match_df[(match_df["season"] == season)
                 & ((match_df["team1"] == team) | (match_df["team2"] == team))]
    return win_rate(team, m)


def get_form(team, n=5):
    m = match_df[(match_df["team1"] == team) | (match_df["team2"] == team)].tail(n)
    return win_rate(team, m)


def get_bat(team):
    m = match_df[match_df["team1"] == team]
    return m["team1_won"].mean() if len(m) else 0.5


def get_chase(team):
    m = match_df[match_df["team2"] == team]
    return (m["team1_won"] == 0).mean() if len(m) else 0.5


def get_h2h(t1, t2):
    m = match_df[((match_df["team1"] == t1) & (match_df["team2"] == t2))
                 | ((match_df["team1"] == t2) & (match_df["team2"] == t1))]
    return win_rate(t1, m)


def get_streak(team):
    return streak(team, match_df)


def get_vwr(team, venue):
    m = match_df[(match_df["venue"] == venue)
                 & ((match_df["team1"] == team) | (match_df["team2"] == team))]
    return win_rate(team, m)


def get_vavg(venue):
    m = match_df[match_df["venue"] == venue]
    if len(m) == 0:
        return 150
    return fi[fi["match_id"].isin(m["match_id"])]["fi_runs"].mean()


def build_features(t1, t2, venue, is_knockout=0):
    return pd.DataFrame([{
        "wr_diff": get_wr(t1) - get_wr(t2),
        "swr_diff": get_swr(t1) - get_swr(t2),
        "f5_diff": get_form(t1, 5) - get_form(t2, 5),
        "f10_diff": get_form(t1, 10) - get_form(t2, 10),
        "bat_diff": get_bat(t1) - get_bat(t2),
        "ch_diff": get_chase(t1) - get_chase(t2),
        "str_diff": get_streak(t1) - get_streak(t2),
        "h2h_diff": get_h2h(t1, t2) - 0.5,
        "t1_vwr": get_vwr(t1, venue),
        "venue_avg": get_vavg(venue),
        "season": 2026,
        "is_knockout": is_knockout,
    }])[FEATURES]


def get_proba(t1, t2, venue, is_knockout=0):
    """Return np.array([P(t2 wins), P(t1 wins)]) with t1 batting first."""
    feat = build_features(t1, t2, venue, is_knockout)
    if best_model_name == "Neural Network":
        nn_model.eval()
        with torch.no_grad():
            p1 = nn_model(torch.FloatTensor(nn_scaler.transform(feat))).item()
        return np.array([1 - p1, p1])
    return models[best_model_name].predict_proba(feat)[0]

In [ ]:
# First 20 fixtures of IPL 2026
fixtures = [
    ("Mar 28", "RCB", "SRH", "Bengaluru"),
    ("Mar 29", "MI", "KKR", "Mumbai"),
    ("Mar 30", "RR", "CSK", "Guwahati"),
    ("Mar 31", "PBKS", "GT", "Mullanpur"),
    ("Apr 1", "LSG", "DC", "Lucknow"),
    ("Apr 2", "KKR", "SRH", "Kolkata"),
    ("Apr 3", "CSK", "PBKS", "Chennai"),
    ("Apr 4", "DC", "MI", "Delhi"),
    ("Apr 4", "GT", "RR", "Ahmedabad"),
    ("Apr 5", "SRH", "LSG", "Hyderabad"),
    ("Apr 5", "RCB", "CSK", "Bengaluru"),
    ("Apr 6", "KKR", "PBKS", "Kolkata"),
    ("Apr 7", "RR", "MI", "Guwahati"),
    ("Apr 8", "DC", "GT", "Delhi"),
    ("Apr 9", "KKR", "LSG", "Kolkata"),
    ("Apr 10", "RR", "RCB", "Guwahati"),
    ("Apr 11", "PBKS", "SRH", "Mullanpur"),
    ("Apr 11", "CSK", "DC", "Chennai"),
    ("Apr 12", "LSG", "GT", "Lucknow"),
    ("Apr 12", "MI", "RCB", "Mumbai"),
]

print(f"{'#':<4} {'Date':<8} {'Bats':<6} {'Chases':<6} {'Bat%':>7} {'Chase%':>8} {'Winner':>10}")
print("=" * 55)

win_counts = {t: 0 for t in team_venues}

for i, (date, tA, tB, city) in enumerate(fixtures, 1):
    venue = city_to_venue[city]
    t1, t2 = (tA, tB) if random.random() > 0.5 else (tB, tA)  # random batting order
    prob = get_proba(t1, t2, venue)
    winner = t1 if prob[1] > 0.5 else t2
    win_counts[winner] += 1
    print(f"{i:<4} {date:<8} {t1:<6} {t2:<6} {prob[1] * 100:>6.1f}% {prob[0] * 100:>7.1f}% {'🏆 ' + winner:>10}")

In [ ]:
# Predicted points table
pt = pd.DataFrame({
    "Team": list(win_counts.keys()),
    "Wins": list(win_counts.values()),
    "Points": [w * 2 for w in win_counts.values()],
}).sort_values("Points", ascending=False).reset_index(drop=True)
pt.index += 1

print("=== PREDICTED POINTS TABLE ===")
print(pt.to_string())
print(f"\nTop 4: {pt['Team'].iloc[:4].tolist()}")

In [ ]:
# Playoff predictor (deterministic: higher-probability team wins)
p1, p2, p3, p4 = pt["Team"].iloc[:4].tolist()


def playoff_predict(tA, tB, stage):
    t1, t2 = (tA, tB) if random.random() > 0.5 else (tB, tA)
    prob = get_proba(t1, t2, team_venues[t1], is_knockout=1)
    winner, loser = (t1, t2) if prob[1] > 0.5 else (t2, t1)
    print(f"  {stage:<30} {tA} vs {tB}  →  🏆 {winner}  ({max(prob) * 100:.1f}% confidence)")
    return winner, loser


print("=" * 60)
print("           IPL 2026 PLAYOFF PREDICTIONS")
print("=" * 60)
print(f"  Top 4 → {p1}, {p2}, {p3}, {p4}\n")

q1_win, q1_los = playoff_predict(p1, p2, "Qualifier 1  (1st vs 2nd)")
el_win, el_los = playoff_predict(p3, p4, "Eliminator   (3rd vs 4th)")
q2_win, q2_los = playoff_predict(q1_los, el_win, "Qualifier 2")
champ, runner = playoff_predict(q1_win, q2_win, "FINAL")

print(f"\n{'=' * 60}")
print(f"  🏆  IPL 2026 PREDICTED CHAMPION :  {champ}")
print(f"  🥈  Runner Up                   :  {runner}")
print(f"{'=' * 60}")

## 10. Monte Carlo Season Simulation

Model probabilities are precomputed once per matchup, then the season is simulated
`N_SIMULATIONS` times by sampling each match outcome from those probabilities.
The 20 fixtures above are a partial schedule, so "Avg Points" and "Top 4 %" reflect
only those matches.

In [ ]:
print("Precomputing match probabilities...")

fixture_probs = {}
for _, tA, tB, city in fixtures:
    venue = city_to_venue[city]
    fixture_probs[(tA, tB, city)] = {
        "AB": get_proba(tA, tB, venue),   # tA bats first -> [P(tB wins), P(tA wins)]
        "BA": get_proba(tB, tA, venue),   # tB bats first -> [P(tA wins), P(tB wins)]
    }

all_teams = list(team_venues.keys())
playoff_probs = {
    (tA, tB): get_proba(tA, tB, team_venues[tA], is_knockout=1)
    for tA in all_teams for tB in all_teams if tA != tB
}
print(f"  ✓ {len(fixture_probs)} fixture probabilities precomputed")
print(f"  ✓ {len(playoff_probs)} playoff matchup probabilities precomputed")

In [ ]:
def fast_sim(tA, tB):
    """Simulate one playoff match using precomputed probabilities."""
    if random.random() > 0.5:
        p_tA = playoff_probs[(tA, tB)][1]   # tA bats first
    else:
        p_tA = playoff_probs[(tB, tA)][0]   # tB bats first
    winner = tA if random.random() < p_tA else tB
    return winner, (tB if winner == tA else tA)


def top_four(sim_wins):
    """Top 4 by wins, with random tie-breaking."""
    ranked = sorted(sim_wins.items(), key=lambda kv: kv[1], reverse=True)
    top4, i = [], 0
    while len(top4) < 4:
        j = i
        while j < len(ranked) and ranked[j][1] == ranked[i][1]:
            j += 1
        group = [ranked[k][0] for k in range(i, j)]
        random.shuffle(group)
        top4.extend(group)
        i = j
    return top4[:4]


champion_counts = defaultdict(int)
finalist_counts = defaultdict(int)
top4_counts = defaultdict(int)
points_accum = defaultdict(int)

start = time.time()
for sim in range(N_SIMULATIONS):
    # League stage
    sim_wins = {t: 0 for t in all_teams}
    for _, tA, tB, city in fixtures:
        probs = fixture_probs[(tA, tB, city)]
        p_tA_wins = probs["AB"][1] if random.random() > 0.5 else probs["BA"][0]
        sim_wins[tA if random.random() < p_tA_wins else tB] += 1

    for t in all_teams:
        points_accum[t] += sim_wins[t] * 2

    top4 = top_four(sim_wins)
    for t in top4:
        top4_counts[t] += 1
    sp1, sp2, sp3, sp4 = top4

    # Playoffs
    q1_w, q1_l = fast_sim(sp1, sp2)     # Qualifier 1
    el_w, _ = fast_sim(sp3, sp4)        # Eliminator
    q2_w, _ = fast_sim(q1_l, el_w)      # Qualifier 2
    champ_sim, _ = fast_sim(q1_w, q2_w) # Final

    finalist_counts[q1_w] += 1
    finalist_counts[q2_w] += 1
    champion_counts[champ_sim] += 1

print(f"✓ {N_SIMULATIONS:,} simulations completed in {time.time() - start:.1f}s")

In [ ]:
mc_df = pd.DataFrame({
    "Team": all_teams,
    "Champion %": [round(champion_counts[t] / N_SIMULATIONS * 100, 1) for t in all_teams],
    "Finalist %": [round(finalist_counts[t] / N_SIMULATIONS * 100, 1) for t in all_teams],
    "Top 4 %": [round(top4_counts[t] / N_SIMULATIONS * 100, 1) for t in all_teams],
    "Avg Points": [round(points_accum[t] / N_SIMULATIONS, 1) for t in all_teams],
}).sort_values("Champion %", ascending=False).reset_index(drop=True)
mc_df.index += 1

print("=" * 60)
print(f"   MONTE CARLO RESULTS — {N_SIMULATIONS:,} SIMULATIONS")
print("=" * 60)
print(mc_df.to_string())
mc_df.to_csv(os.path.join(OUTPUT_DIR, "monte_carlo_results.csv"))

In [ ]:
sorted_teams = mc_df["Team"].tolist()
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.9, len(sorted_teams)))

panels = [
    ("Champion %", "🏆 Championship Probability", "Probability (%)", 0.2),
    ("Finalist %", "🥈 Finalist Probability", "Probability (%)", 0.2),
    ("Top 4 %", "🔝 Top 4 Probability", "Probability (%)", 0.2),
    ("Avg Points", "📊 Average Predicted Points", "Average Points", 0.1),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f"IPL 2026 — Monte Carlo Simulation ({N_SIMULATIONS:,} runs)",
             fontsize=15, fontweight="bold", y=1.01)

for ax, (col, title, xlabel, pad) in zip(axes.flat, panels):
    vals = mc_df.set_index("Team").loc[sorted_teams, col]
    bars = ax.barh(sorted_teams[::-1], vals[::-1], color=colors[::-1])
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontweight="bold")
    for bar, val in zip(bars, vals[::-1]):
        ax.text(bar.get_width() + pad, bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "monte_carlo.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print(f"\n{'=' * 55}")
print(f"  🏆  MOST LIKELY IPL 2026 CHAMPION : {mc_df.iloc[0]['Team']}")
print(f"  📊  Championship Probability      : {mc_df.iloc[0]['Champion %']}%")
print(f"  🥈  Most Likely Finalist          : {mc_df.sort_values('Finalist %', ascending=False).iloc[0]['Team']}")
print("\n  Top 4 most likely teams:")
for _, row in mc_df.sort_values("Top 4 %", ascending=False).head(4).iterrows():
    print(f"    {row['Team']:<6}  Champion: {row['Champion %']:>5}%  |  Top4: {row['Top 4 %']:>5}%")
print(f"{'=' * 55}")